# Phase 9.1 — Context-Aware WaveToText (Google Colab)

**FLUX Architecture** — Field-based Latent Understanding eXperience

## Overview
Phase 9.1 replaces the Phase 9 **WaveToText** decoder with a new **ContextWaveToText** module.
It uses cross-attention over up to 2 prior left-context chunks to resolve spelling ambiguities.
Frozen components: CSE, WaveChunker, Field, Generator, etc. We train *only* the new decoder.

## Cell 1 — Clone Repo & Install Dependencies

In [ ]:
# Cell 1 — Clone / pull FLUX repo and install dependencies
import os, subprocess

REPO_URL = "https://github.com/Unseengap/FLUX.git"
REPO_DIR = "/content/FLUX"

if os.path.exists(REPO_DIR):
    print("  ℹ Repo exists — pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    print("  ℹ Cloning FLUX repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f"  ✓ Working directory: {os.getcwd()}")

# Fix faiss-gpu issue on Colab by removing it from requirements temporarily
if os.path.exists("requirements.txt"):
    with open("requirements.txt", "r") as f:
        reqs = f.read()
    reqs = reqs.replace("faiss-gpu>=1.7.4", "faiss-cpu>=1.7.4")
    with open("requirements.txt", "w") as f:
        f.write(reqs)

# Install dependencies (suppress pip output)
subprocess.run(
    ["pip", "install", "-q", "-e", ".", "--no-deps"],
    check=False, capture_output=True,
)
subprocess.run(
    ["pip", "install", "-q", "-r", "requirements.txt"],
    check=False, capture_output=True,
)
# Ensure datasets is available
subprocess.run(
    ["pip", "install", "-q", "datasets", "huggingface_hub"],
    check=False, capture_output=True,
)
print("  ✓ Dependencies installed")

## Cell 2 — Hardware, Secrets, Logger

In [ ]:
# Cell 2 — Hardware, Secrets, Logger, Config
import sys, torch, time
from pathlib import Path

# Path setup
REPO_DIR = Path("/content/FLUX")
for _p in ['phases/phase1', 'phases/phase2', 'phases/phase3',
           'phases/phase4', 'phases/phase5', 'phases/phase6',
           'phases/phase7', 'phases/phase8', 'phases/phase9', 'phases/phase9_1']:
    _pp = str(REPO_DIR / _p)
    if _pp not in sys.path:
        sys.path.insert(0, _pp)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from flux_utils import get_device, PhaseLogger, PhaseResults

# Hardware
device = get_device()
print(f"  Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM: {_vram:.1f} GB")

# Secrets — HuggingFace token from Colab
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print(f"  ✓ HF_TOKEN loaded ({len(HF_TOKEN)} chars)")
except Exception:
    import os
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if HF_TOKEN:
        print(f"  ✓ HF_TOKEN from env ({len(HF_TOKEN)} chars)")
    else:
        print("  ⚠ No HF_TOKEN — checkpoint upload will be skipped")

# Logger
log = PhaseLogger(phase=9)
log.separator("Phase 9.1: Context-Aware WaveToText")
log.cell_start("Cell 2 — Hardware & Secrets")
log.info(f"Device: {device}")
from train_context_wtt import PHASE9_1_CONFIG
print("  Phase 9.1 Config:")
for k, v in PHASE9_1_CONFIG.items():
    print(f"    {k}: {v}")
log.cell_end("Cell 2 — Hardware & Secrets", "PASS")

## Cell 3 — Smoke Test

In [ ]:
# Cell 3 — Smoke Test: Verify loading Phase 9 states and building ContextWTT
log.cell_start("Cell 3 — Smoke Test")
from train_context_wtt import load_phase9_for_9_1, build_context_wtt

# 1. Load frozen components
print("  ℹ Loading Phase 9 checkpoint for frozen components...")
cse, chunker, ckpt9 = load_phase9_for_9_1(device=device)

# Test CSE
test_wave = cse.encode("hello world")
assert test_wave.full.shape[-1] == 432, f"Expected 432, got {test_wave.full.shape[-1]}"
log.success(f"CSE smoke test: {test_wave.full.shape}")

# Test Chunker
wave_seq = test_wave.full.to(device)
chunks, spans = chunker(wave_seq)
log.success(f"WaveChunker: {wave_seq.shape[0]} waves → {chunks.shape[0]} chunks")

# 2. Build fresh ContextWaveToText
context_wtt = build_context_wtt(device=device)

# Test forward
test_bytes = torch.tensor([104, 101, 108, 108, 111], dtype=torch.long, device=device)
with torch.no_grad():
    logits = context_wtt(chunks[0].unsqueeze(0), test_bytes.unsqueeze(0))
assert logits.shape[:2] == (1, 6), f"Expected (1, 6, ...), got {logits.shape}"
log.success(f"ContextWTT forward: logits {logits.shape}")

log.cell_end("Cell 3 — Smoke Test", "PASS")
print("\n  ✓ Smoke test passed — ready for dataset prep")

## Cell 4 — Load Training Data

In [ ]:
# Cell 4 — Load Data
log.cell_start("Cell 4 — Load Data")
from train_context_wtt import load_training_data, PHASE9_1_CONFIG

texts = load_training_data(max_docs=PHASE9_1_CONFIG['max_train_docs'])
split_idx = int(len(texts) * 0.9)
train_texts = texts[:split_idx]
eval_texts  = texts[split_idx:]
print(f"  Train: {len(train_texts):,} docs | Eval: {len(eval_texts):,} docs")
log.cell_end("Cell 4 — Load Data", "PASS")

## Cell 5 — Prepare Training Pairs

In [ ]:
# Cell 5 — Prepare training pairs (current_wave, context_waves, bytes)
log.cell_start("Cell 5 — Prep Pairs")
from train_context_wtt import Phase9_1_Trainer, UTF8_AUGMENT_TEXTS

trainer = Phase9_1_Trainer(
    cse=cse,
    chunker=chunker,
    context_wtt=context_wtt,
    device=device,
    log=log,
)

training_pairs = trainer.prepare_training_data(
    train_texts,
    max_pairs=500000,
    utf8_texts=UTF8_AUGMENT_TEXTS,
)
log.metric("training_pairs", f"{len(training_pairs):,}")
log.cell_end("Cell 5 — Prep Pairs", "PASS")

## Cell 6 — Train ContextWaveToText

In [ ]:
# Cell 6 — Train ContextWaveToText
log.cell_start("Cell 6 — Train CWTT")

train_result = trainer.train(
    training_pairs,
    max_steps=PHASE9_1_CONFIG['max_steps'],
    batch_size=PHASE9_1_CONFIG['batch_size'],
    grad_accum=PHASE9_1_CONFIG['grad_accum'],
    lr=PHASE9_1_CONFIG['lr'],
    warmup_steps=PHASE9_1_CONFIG['warmup_steps'],
    log_interval=PHASE9_1_CONFIG['log_interval'],
    eval_interval=PHASE9_1_CONFIG['eval_interval'],
    eval_texts=eval_texts,
)
log.cell_end("Cell 6 — Train CWTT", "PASS")

## Cell 7 — Diagnostics

In [ ]:
# Cell 7 — Diagnostics
log.cell_start("Cell 7 — Diagnostics")

print("  📊 Chunk-Level Accuracy")
chunk_acc = trainer._evaluate_chunk_accuracy(eval_texts[:50], max_chunks=500)
print(f"  Chunk accuracy: {chunk_acc:.1%} (Phase 9 baseline: 82.8%)")

print("\n  📊 Hard Spelling Test")
from train_context_wtt import HARD_SPELLING_WORDS
hard_acc = trainer._evaluate_hard_spelling(HARD_SPELLING_WORDS)
print(f"  Hard spelling: {hard_acc:.0%} (Phase 9 baseline: 33%)")

print("\n  📊 UTF-8 Multi-Byte")
utf8_acc = trainer._evaluate_utf8_accuracy()
print(f"  UTF-8 accuracy: {utf8_acc:.0%} (Phase 9 baseline: 0%)")

log.cell_end("Cell 7 — Diagnostics", "PASS")

## Cell 8 — Save & Upload Checkpoint

In [ ]:
# Cell 8 — Save Checkpoint
log.cell_start("Cell 8 — Save Checkpoint")

ckpt_path = trainer.save_checkpoint(train_result, ckpt9)
print(f"  ✓ Saved cleanly to {ckpt_path}")

if HF_TOKEN:
    from flux_utils import upload_checkpoint_to_hf
    try:
        upload_checkpoint_to_hf(phase=9.1, hf_token=HF_TOKEN)
        print("  ✓ Checkpoint uploaded to HuggingFace")
    except Exception as e:
        print(f"  ⚠ Checkpoint upload failed: {e}")

log.cell_end("Cell 8 — Save Checkpoint", "PASS")

---
## Tests
Run all three Phase 9.1 tests. Each is a standalone script.

In [ ]:
# Cell 9 — Test 1
log.cell_start("Cell 9 — Test 1")
!cd /content/FLUX && python phases/phase9_1/test_phase9_1_test1.py
log.cell_end("Cell 9 — Test 1")

In [ ]:
# Cell 10 — Test 2
log.cell_start("Cell 10 — Test 2")
!cd /content/FLUX && python phases/phase9_1/test_phase9_1_test2.py
log.cell_end("Cell 10 — Test 2")

In [ ]:
# Cell 11 — Test 3
log.cell_start("Cell 11 — Test 3")
!cd /content/FLUX && python phases/phase9_1/test_phase9_1_test3.py
log.cell_end("Cell 11 — Test 3")

---
## Demos
Run Phase 9.1 demo scripts.

In [ ]:
# Cell 12 — Demo 1: Spelling Comparison
log.cell_start("Cell 12 — Demo 1")
!cd /content/FLUX && python phases/phase9_1/demo_phase9_1_demo1.py
log.cell_end("Cell 12 — Demo 1")

In [ ]:
# Cell 13 — Demo 2: Context Visualization
log.cell_start("Cell 13 — Demo 2")
!cd /content/FLUX && python phases/phase9_1/demo_phase9_1_demo2.py
log.cell_end("Cell 13 — Demo 2")

## Cell 14 — Interactive

In [ ]:
# Cell 14 — Interactive Decoding
log.cell_start("Cell 14 — Interactive")
text = "The overarching implication here is that causality and temporal semantics are intertwined intrinsically."
print(f"Original text:\n{text}\n")

try:
    w = cse.encode(text).full.to(device)
    cw, sp = chunker(w)
    dec = context_wtt.decode_sequence(cw, temperature=0.3)
    dec_str = b''.join(dec).decode('utf-8', errors='replace')
    print(f"Reconstructed via ContextWaveToText:\n{dec_str}")
except Exception as e:
    print(f"Error: {e}")
log.cell_end("Cell 14 — Interactive", "PASS")

## Cell 15 — View Results

In [ ]:
# Cell 15 — Generate and view results
log.cell_start("Cell 15 — Results")

results = PhaseResults(phase=9, component_name="Context-Aware WaveToText (9.1)")
results.add_test("Chunk Accuracy", train_result.chunk_accuracy >= 0.93, score=f"{train_result.chunk_accuracy:.1%}", threshold="≥93%")
results.add_test("Word Accuracy", train_result.word_accuracy >= 0.88, score=f"{train_result.word_accuracy:.1%}", threshold="≥88%")
results.add_test("Hard Spelling", train_result.hard_spelling_accuracy >= 0.60, score=f"{train_result.hard_spelling_accuracy:.0%}", threshold="≥60%")
results.add_test("UTF-8 Accuracy", train_result.utf8_accuracy >= 0.50, score=f"{train_result.utf8_accuracy:.0%}", threshold="≥50%")
results.add_metric("Training steps", train_result.total_steps)
results.add_metric("Total time", f"{train_result.total_time_seconds:.0f}s")
res_path = Path("/content/FLUX/phases/phase9_1/RESULTS_PHASE_9_1.md")
results.save(path=str(res_path))

if res_path.exists():
    print(res_path.read_text())
log.cell_end("Cell 15 — Results", "PASS")

## Cell 16 — View Log

In [ ]:
# Cell 16 — View full training log
_logpath = Path("/content/FLUX/logs/phase9.log")
if _logpath.exists():
    _lines = _logpath.read_text().splitlines()
    print(f"  Log: {len(_lines)} lines")
    # Print last 50 lines
    for line in _lines[-50:]:
        print(line)
else:
    print("  ⚠ No log file found")

## Cell 17 — Upload Logs

In [ ]:
# Cell 17 — Upload Logs
log.cell_start("Cell 17 — Upload Logs")
if HF_TOKEN:
    from flux_utils import upload_logs_to_hf
    try:
        upload_logs_to_hf(phase=9, hf_token=HF_TOKEN)
        print("  ✓ Logs uploaded to HuggingFace")
    except Exception as e:
        print(f"  ⚠ Log upload failed: {e}")
log.cell_end("Cell 17 — Upload Logs", "PASS")

## Cell 18 — Save Colab Artifacts

In [ ]:
# Cell 18 — Copy artifacts to /content/output
import shutil
_out = Path("/content/output")
_out.mkdir(exist_ok=True)

# Checkpoint
_ckpt = Path("/content/FLUX/checkpoints/phase9_1.phase.pt")
if _ckpt.exists():
    shutil.copy2(_ckpt, _out / "phase9_1.phase.pt")
    print(f"  ✓ Checkpoint → {_out / 'phase9_1.phase.pt'}")

# Results
_res = Path("/content/FLUX/phases/phase9_1/RESULTS_PHASE_9_1.md")
if _res.exists():
    shutil.copy2(_res, _out / "RESULTS_PHASE_9_1.md")
    print(f"  ✓ Results → {_out / 'RESULTS_PHASE_9_1.md'}")

# Log
_log = Path("/content/FLUX/logs/phase9.log")
if _log.exists():
    shutil.copy2(_log, _out / "phase9.log")
    print(f"  ✓ Log → {_out / 'phase9.log'}")

print(f"\n  All artifacts saved to {_out}")